# Phase 1: Background margin study

## Section 1 - Purpose and overview

This study chooses the background margin for the two classifiers kept so far: V0, one prototype per class, and V2, every labelled reference patch as a prototype (multi-prototype study). It also asks whether V2, with its own margin, keeps its advantages and recovers what it lost in the continuous perception of assembly_2.

What the margin does. Each patch has a similarity with every class. Let d be the similarity of the best class other than background minus the similarity of background. The patch takes that class if d is at least the margin m, and background otherwise. The margin decides only between background and the other classes, the pieces and the hand. It never decides between two pieces.

The margin is a bias on the background class. Mensink et al. (2013, Sect. 3.4, Eq. 21) write the nearest class mean classifier as a dot product plus a bias for each class, and these biases set the prior probabilities of the classes. With m = 0 each patch takes its most similar class. With equal costs for all errors, choosing the most probable class minimizes the number of errors (Bishop 2006, Sect. 1.5.1). When one error costs more than another, the decision boundary moves to avoid the costlier one (Bishop 2006, Sect. 1.5.2). The direction depends only on which error costs more. A positive margin favours background, a negative margin favours the other classes. Until now the margin could only be zero or positive. From this study, negative values are allowed (assign_labels, src/perception/prototypes.py).

Why the margin is studied now. The current value, 0.03, was set by eye for V0 and never studied (pipeline, Section 9). On the validation frames it raises the mIoU of V0 with the arms merged from 0.670 to 0.725, while V2 barely changes, from 0.864 to 0.861 (multi-prototype study, Observation 5.2). In the continuous perception of assembly_2, V2 rarely saw the loose arm_2 on the table: in 10% of the frames before its pick, against 99% with V0. At rest, the presence of arm_2 changed about 5 times per second with V2, against 0.2 with V0 (multi-prototype study, Addendum; continuous_analysis.py on the saved labels).

Hypothesis, not verified: with 37,864 background prototypes, background often wins by a small difference, and a lower or negative margin would recover these patches. A margin of zero or above cannot recover a patch where background already has the highest similarity.

Where the margin is chosen. Choosing the margin on the validation frames would make their results optimistic (Cawley and Talbot 2010, Sect. 4.4 and 5; Bishop 2006, Sect. 1.3). It is chosen instead on the 10 reference frames, by cross-validation (Bishop 2006, Sect. 1.3). One reference sequence at a time is left out. The prototypes are built from the other frames, and the frames left out are classified and compared with their annotations.

Whole sequences are left out, not single frames. Frames of one recording look alike, and for V2 a nearly identical frame among the prototypes would make the task too easy. The same reason splits validation and test by recording (Sener et al. 2022, Sect. 4.6).

The selection needs no annotation beyond the reference frames, which the method needs anyway. In a new assembly it can be repeated automatically. Cawley and Talbot (2010, Sect. 6) recommend treating model selection as part of fitting, also in operational use.

Details of the selection. The hand prototypes come from one frame only, caos_f0211. In the fold that leaves out the caos sequence there is no hand class, so the hand patches of that fold are ignored. The mIoU changes only when the margin crosses the d of some patch. It is therefore computed at every value where at least one label changes, with no grid to choose, and zero and negative values are included. The chosen margin m* is the middle of the best interval, the value farthest from both of its ends. If two separate intervals tie, the one closest to zero is taken, since zero adds no preference. The margin is chosen separately for V0 and V2, because a class mean and a maximum over many prototypes give similarities on different scales (Section 4).

Hypotheses, written before running. For V0, m* is positive, since 0.03 already improves V0 on validation. For V2, m* is lower than for V0, possibly negative.

Decision rule, fixed before running. For each variant, the mIoU at m* is compared with the mIoU at margin 0 on the validation frames, with the arms merged on the main metric, by the paired 95% bootstrap interval of the difference (Koehn 2004, Sect. 6), on the same resamples as the previous studies. If the interval lies entirely below zero, the margin is 0, otherwise m*. Zero is the reference because it is the decision without any preference between background and pieces. The value 0.03 is reported only. Then V2 is compared with V0, each at its chosen margin. If the interval lies entirely above zero, the choice of V2 is confirmed. Otherwise the study stops and the result is discussed before any change to the pipeline.

Reported, outside the rule. The mIoU over the 10 pieces and with the boundary ring ignored, the per-class IoU, and three kinds of error separately: background predicted as a piece, near the pieces or far from them; a piece predicted as background; a piece predicted as another piece. Also the targeted measures of the multi-prototype study: the empty mouth slot, the identity of the arms, the loose mouth. If the chosen margin visibly increases the false positives far from the pieces, this is discussed before updating the pipeline.

After the decision, as an exploratory check outside the rule, continuous perception is run again on assembly_2 for V0 and V2 at 0.03 and at the chosen margins, and analysed with continuous_analysis.py unchanged. Its thresholds are provisional. A diagnostic also asks why arm_2 is missed: whether background wins outright, or the margin rejects it. The test sequence (assembly_1) is not read.

Section 2 sets the configuration. Section 3 checks that the new assign_labels reproduces the previous results. Section 4 describes d for both variants. Section 5 chooses the margin on the reference frames. Section 6 applies the decision rule on the validation frames. Section 7 reports the errors and the targeted measures. Section 8 is the continuous perception. Section 9 gives the conclusions.

------------------------------------------------------------------------------------------------------------------------------------------

References Section 1:

Mensink et al. 2013, Sect. 3.4, Eq. (21): the nearest class mean classifier written with the dot product and a class-specific bias; the biases define the prior probabilities of the classes.

Bishop 2006, Pattern Recognition and Machine Learning, Sect. 1.5.1: the number of errors is smallest when each input is assigned to the class with the largest posterior probability. Sect. 1.5.2, Eq. (1.80) and (1.81): with a loss matrix, each input is assigned to the class that minimizes the expected loss, so errors of different cost move the decision boundary. Sect. 1.3: when a design is iterated many times on a small validation set, it can over-fit that set; cross-validation (Figure 1.18) uses part of the data to build the model and the rest to evaluate it, in turn.

Cawley and Talbot 2010, Sect. 4.4: optimizing a selection criterion on a finite sample over-fits, more so with small samples and many choices. Sect. 5: performance estimated on the data used for selection is optimistically biased; selection must be treated as part of fitting. Sect. 6: model selection should be repeated independently in each trial, which also reflects practice in operational use.

Sener et al. 2022, Assembly101, Sect. 4.6: splits made by recording.

Koehn 2004, Sect. 6: paired bootstrap resampling.

------------------------------------------------------------------------------------------------------------------------------------------

## Section 2 - Configuration

All paths, frames and hyperparameters in one place. The values are those of the multi-prototype study (Section 2.4 and 2.5), with the same 10 reference frames and 8 validation frames. Only the margin changes: 0 is the reference of the decision rule, and 0.03 is kept only to be reported. The cross-validation folds are the reference sequences.

In [1]:
# 2.1 Dependencies: install once per environment with pip install -r requirements.txt (README, Setup)

In [2]:
# 2.2 Imports
%load_ext autoreload
%autoreload 2

import sys
import json
import hashlib
import platform
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import transformers
import matplotlib.pyplot as plt
from PIL import Image
from scipy import ndimage

In [3]:
# 2.3 Paths and shared module
PROJECT_ROOT = Path.cwd().parent
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))   # makes src/perception importable from notebooks/

MODEL_DIR = PROJECT_ROOT / "models" / "c_radio_weights"
FRAMES_DIR = PROJECT_ROOT / "data" / "extracted_frames"
VIDEO_PATH = PROJECT_ROOT / "data" / "raw_videos" / "assembly_2.mp4"              # Section 8
EVENTS_PATH = PROJECT_ROOT / "configs" / "assembly_2_events.json"                 # Section 8
SCRATCH_DIR = PROJECT_ROOT / "outputs" / "scratch"                                # labels at 0.03 of the Addendum (Section 8)
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "phase_1_margin_study"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

for name, path in {"MODEL_DIR": MODEL_DIR, "FRAMES_DIR": FRAMES_DIR, "SRC_DIR": SRC_DIR,
                   "VIDEO_PATH": VIDEO_PATH, "EVENTS_PATH": EVENTS_PATH,
                   "labels V0 (0.03)": SCRATCH_DIR / "assembly_2_continuous_labels.npz",
                   "labels V2 (0.03)": SCRATCH_DIR / "assembly_2_continuous_labels_V2.npz"}.items():
    status = "OK" if path.exists() else "MISSING"
    print(f"[{status}] {name}: {path}")

# Shared code (src/perception)
from perception.classes import CLASS_NAMES, BACKGROUND_ID, HAND_LABEL, PIECE_LABELS, IGNORE_ID, PATCH_LABEL_NAMES
from perception.features import FeatureExtractor
from perception.labels import sequence_of, check_annotation, rasterize_annotation, patch_label_grid
from perception.prototypes import (build_prototypes, stack_prototypes, patch_prototypes, class_similarities,
                                   class_similarities_gpu, sort_by_class, assign_labels, l2_normalize)
from perception.metrics import (confusion_matrix, iou_from_confusions, mean_iou, ignore_boundary_ring,
                                MERGED_NAMES, MERGED_PIECES, merge_arms, merged_ids,
                                frame_iou, frame_miou, bootstrap_miou, bootstrap_class_iou, percentile_interval)

[OK] MODEL_DIR: c:\Users\ADV\Universita\Magistrale\Tesi\vad-project\models\c_radio_weights
[OK] FRAMES_DIR: c:\Users\ADV\Universita\Magistrale\Tesi\vad-project\data\extracted_frames
[OK] SRC_DIR: c:\Users\ADV\Universita\Magistrale\Tesi\vad-project\src
[OK] VIDEO_PATH: c:\Users\ADV\Universita\Magistrale\Tesi\vad-project\data\raw_videos\assembly_2.mp4
[OK] EVENTS_PATH: c:\Users\ADV\Universita\Magistrale\Tesi\vad-project\configs\assembly_2_events.json
[OK] labels V0 (0.03): c:\Users\ADV\Universita\Magistrale\Tesi\vad-project\outputs\scratch\assembly_2_continuous_labels.npz
[OK] labels V2 (0.03): c:\Users\ADV\Universita\Magistrale\Tesi\vad-project\outputs\scratch\assembly_2_continuous_labels_V2.npz


In [4]:
# 2.4 Hyperparameters
# Same values as the multi-prototype study (Section 2.4); only the background margin is new.

# --- Feature extraction ---
DEVICE = "cuda"
DTYPE = torch.bfloat16          # input resolution: native 1024x1024 (resolution study)

# --- Patch labelling ---
MIN_LABEL_COVERAGE = 0.5        # same rule as the pipeline (Section 5)

# --- Background margin ---
NO_MARGIN = 0.0                 # no preference between background and the other classes: reference of the rule (Section 1)
CURRENT_MARGIN = 0.03           # set by eye for V0 (pipeline, Section 9): reported only, not a candidate
PROTOTYPE_CHUNK = 4096          # prototypes compared at once by class_similarities on the CPU: memory only, results unchanged

# --- Evaluation ---
N_BOOTSTRAP = 1000              # resamples of the frames (support-set study, Section 5.4)
BOUNDARY_RING = 1               # secondary metric: background patches within 1 patch of an object are ignored

# --- Reproducibility ---
RANDOM_SEED = 0
np.random.seed(RANDOM_SEED)
_ = torch.manual_seed(RANDOM_SEED)

In [5]:
# 2.5 Frames and split
# Validation: the chosen margins are checked here (decision rule, Section 1).
# Test: held out until the end of all studies. This notebook never reads a test frame.
VALIDATION_SEQUENCES = {"assembly_2", "1_disassembled_controlled_caos"}
TEST_SEQUENCES = {"assembly_1"}
assert not VALIDATION_SEQUENCES & TEST_SEQUENCES


def frame_path(frame, suffix=".jpg"):
    """Path of a frame or of its annotation. Refuses test frames, so the test set cannot be read by mistake."""
    assert sequence_of(frame) not in TEST_SEQUENCES, f"{frame} belongs to the held-out test set"
    return FRAMES_DIR / f"{frame}{suffix}"


# Support set of 10 frames, in the order of the multi-prototype study (Section 2.5)
REFERENCE_FRAMES = [
    {"frame": "5_assembled_f0002",            "role": "assembled",                   "use_labels": PIECE_LABELS},
    {"frame": "caos_f0149",                   "role": "partially assembled",         "use_labels": PIECE_LABELS},
    {"frame": "5_assembled_f0019",            "role": "assembled, 3/4 view",         "use_labels": PIECE_LABELS},
    {"frame": "3_disassembled_ordered_f0001", "role": "disassembled",                "use_labels": PIECE_LABELS - {"base"}},
    {"frame": "2_disassembled_caos_f0016",    "role": "disassembled, top-down view", "use_labels": PIECE_LABELS - {"base"}},
    {"frame": "Take_3_1_Base_f0099",          "role": "base only, upright",          "use_labels": {"base"}},
    {"frame": "Take_3_1_Base_f0131",          "role": "base only, lying",            "use_labels": {"base"}},
    {"frame": "caos_f0211",                   "role": "hands",                       "use_labels": {HAND_LABEL}},
    {"frame": "4_partially_assembled_f0002",  "role": "body on base, face incomplete (empty mouth slot); mouth loose at 3/4",
     "use_labels": {"body", "mouth"}},
    {"frame": "caos_f0043",                   "role": "mouth loose, side view",      "use_labels": {"mouth"}},
]

# Validation frames in the order of the previous studies: bootstrap resamples pick frames by position
VALIDATION_FRAMES = [
    {"frame": "assembly_2_f0003", "stage": "all pieces loose"},
    {"frame": "assembly_2_f0017", "stage": "body on base"},
    {"frame": "assembly_2_f0030", "stage": "hands: piece held in hand"},
    {"frame": "assembly_2_f0046", "stage": "face (eyes, nose, moustache, mouth)"},
    {"frame": "assembly_2_f0057", "stage": "ears mounted"},
    {"frame": "assembly_2_f0068", "stage": "arms mounted"},
    {"frame": "1_disassembled_controlled_caos_f0001", "stage": "disassembled, front view", "exclude": {"base"}},
    {"frame": "1_disassembled_controlled_caos_f0016", "stage": "disassembled, side view",  "exclude": {"base"}},
]

# No leakage: reference frames never come from validation or test sequences
for ref in REFERENCE_FRAMES:
    assert sequence_of(ref["frame"]) not in VALIDATION_SEQUENCES | TEST_SEQUENCES, f"Leakage: {ref['frame']}"
    ref["image_path"], ref["json_path"] = frame_path(ref["frame"]), frame_path(ref["frame"], ".json")
for val in VALIDATION_FRAMES:
    assert sequence_of(val["frame"]) in VALIDATION_SEQUENCES, f"{val['frame']} is not a validation frame"
    assert frame_path(val["frame"]).exists(), f"Missing frame: {val['frame']}"

GT_LABELS = PIECE_LABELS | {HAND_LABEL}   # validation ground truth: every official class; legacy labels become ignored
EXCLUDED = [{CLASS_NAMES.index(name) for name in val.get("exclude", set())} for val in VALIDATION_FRAMES]

# Cross-validation folds (Section 1): one reference sequence per fold, frames in support-set order
FOLDS = {}
for ref in REFERENCE_FRAMES:
    FOLDS.setdefault(sequence_of(ref["frame"]), []).append(ref["frame"])
assert len(FOLDS) == 6, f"Expected 6 reference sequences, found {len(FOLDS)}"

print(f"{len(REFERENCE_FRAMES)} reference frames in {len(FOLDS)} folds, {len(VALIDATION_FRAMES)} validation frames")
display(pd.DataFrame({"frames left out": [", ".join(frames) for frames in FOLDS.values()],
                      "n": [len(frames) for frames in FOLDS.values()]}, index=list(FOLDS)))

10 reference frames in 6 folds, 8 validation frames


,frames left out,n
5_assembled,"5_assembled_f0002, 5_assembled_f0019",2
caos,"caos_f0149, caos_f0211, caos_f0043",3
3_disassembled_ordered,3_disassembled_ordered_f0001,1
2_disassembled_caos,2_disassembled_caos_f0016,1
Take_3_1_Base,"Take_3_1_Base_f0099, Take_3_1_Base_f0131",2
4_partially_assembled,4_partially_assembled_f0002,1


Observation 2.5: The 10 reference frames form 6 folds, one per sequence. The largest fold is caos, with 3 frames, and it is the only one that contains hand (caos_f0211). In that fold there is no hand prototype, so the hand patches are ignored (Section 1). Three folds have a single frame: 3_disassembled_ordered, 2_disassembled_caos and 4_partially_assembled.

In [6]:
# 2.6 Annotation fingerprints: data/ is not tracked by git, so the notebook records which annotations it used
def fingerprint(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()[:12]

display(pd.DataFrame({
    "set": [f"reference ({sequence_of(ref['frame'])})" for ref in REFERENCE_FRAMES] + ["validation"] * len(VALIDATION_FRAMES),
    "annotation sha256": [fingerprint(ref["json_path"]) for ref in REFERENCE_FRAMES]
                         + [fingerprint(frame_path(val["frame"], ".json")) for val in VALIDATION_FRAMES],
}, index=[ref["frame"] for ref in REFERENCE_FRAMES] + [val["frame"] for val in VALIDATION_FRAMES]))

,set,annotation sha256
5_assembled_f0002,reference (5_assembled),e85d1c179582
caos_f0149,reference (caos),14f88d7cf080
5_assembled_f0019,reference (5_assembled),1cd8cdcc2c9b
3_disassembled_ordered_f0001,reference (3_disassembled_ordered),319c3601f0a1
2_disassembled_caos_f0016,reference (2_disassembled_caos),f17ab6af8218
Take_3_1_Base_f0099,reference (Take_3_1_Base),853e60024c17
Take_3_1_Base_f0131,reference (Take_3_1_Base),f07f67aae3a9
caos_f0211,reference (caos),3c56b4804e54
4_partially_assembled_f0002,reference (4_partially_assembled),32a94339f2b4
caos_f0043,reference (caos),3b555dd93c2c


Observation 2.6: All 18 annotation fingerprints, 10 reference and 8 validation frames, are identical to those recorded by the multi-prototype study (Section 2.6). This study uses exactly the same annotations.

In [7]:
# 2.7 Environment
print(f"Python       : {platform.python_version()}")
print(f"PyTorch      : {torch.__version__}")
print(f"Transformers : {transformers.__version__}")
print(f"CUDA         : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU          : {torch.cuda.get_device_name(0)}")

Python       : 3.10.0
PyTorch      : 2.5.1+cu121
Transformers : 5.17.0
CUDA         : True
GPU          : NVIDIA GeForce RTX 4060 Laptop GPU


------------------------------------------------------------------------------------------------------------------------------------------

References Section 2: none. The values and frames are those of the multi-prototype study (Section 2.4 and 2.5); the folds follow the cross-validation by sequence of Section 1.

------------------------------------------------------------------------------------------------------------------------------------------

## Section 3 - Regression check

From this study, assign_labels also accepts negative margins (Section 1). For margins of zero or above its code is the one of v1.5, so the labels should not change. This section checks it, so that later differences come from the margin and not from the code.

Three things are checked. First, at margin 0 and 0.03 the new assign_labels gives exactly the labels of the v1.5 function, copied here, for V0 and V2 on every validation patch. Second, the mIoU of V0 and V2 at these margins equals the one reported by the multi-prototype study (Observation 5.2). Third, at negative margins the labels follow the definition of Section 1: a patch takes the best class other than background if d is at least the margin, and background otherwise. The negative margins of this check are taken from the data, as quantiles of the negative values of d. They only exercise the code and are not candidates.

The features of the 10 reference frames and of the 8 validation frames are extracted here once, and the ground truth is prepared as in the multi-prototype study (Section 3.3). Class similarities are computed on the CPU with class_similarities, the code used by the multi-prototype study for its measures, so the comparison is with the same computation.

In [8]:
# 3.1 Loading the model (shared FeatureExtractor, same settings as the pipeline)
extractor = FeatureExtractor(MODEL_DIR, device=DEVICE, dtype=DTYPE)
PATCH_SIZE = extractor.patch_size
print(f"Model loaded, patch size {PATCH_SIZE}x{PATCH_SIZE}")

Loading weights:   0%|          | 0/330 [00:00<?, ?it/s]

Model loaded, patch size 16x16


In [9]:
# 3.2 Reference frames: features and patch labels (use_labels); the two variants on all 10 frames
for ref in REFERENCE_FRAMES:
    check_annotation(ref["image_path"], ref["json_path"])
    image = Image.open(ref["image_path"]).convert("RGB")
    ref["features"], size = extractor.extract(image)
    ref["patch_labels"], _ = patch_label_grid(rasterize_annotation(ref["json_path"], ref["use_labels"], size),
                                              PATCH_SIZE, MIN_LABEL_COVERAGE)

prototypes_v0, _ = build_prototypes(REFERENCE_FRAMES)
VARIANTS = {
    "V0": (stack_prototypes(prototypes_v0), np.arange(len(CLASS_NAMES))),   # one prototype per class, in class order
    "V2": patch_prototypes(REFERENCE_FRAMES),                              # every labelled reference patch
}
display(pd.DataFrame({name: {"prototypes": len(owners), "background prototypes": int((owners == BACKGROUND_ID).sum())}
                      for name, (prototypes, owners) in VARIANTS.items()}).T)

,prototypes,background prototypes
V0,12,1
V2,39125,37864


In [10]:
# 3.3 Validation frames: ground truth (multi-prototype study, Section 3.3), features, and class similarities
# of both variants on the CPU (class_similarities, as in the multi-prototype study, Section 5.1)
for val in VALIDATION_FRAMES:
    image_path, json_path = frame_path(val["frame"]), frame_path(val["frame"], ".json")
    check_annotation(image_path, json_path)
    image = Image.open(image_path).convert("RGB")
    val["features"], size = extractor.extract(image)
    val["gt_labels"], _ = patch_label_grid(rasterize_annotation(json_path, GT_LABELS, size), PATCH_SIZE, MIN_LABEL_COVERAGE)
    val["gt_tolerant"] = ignore_boundary_ring(val["gt_labels"], BOUNDARY_RING)

SIMILARITIES = {name: [class_similarities(val["features"], prototypes, owners, chunk=PROTOTYPE_CHUNK)
                       for val in VALIDATION_FRAMES]
                for name, (prototypes, owners) in VARIANTS.items()}
print("Similarities computed:", {name: len(s) for name, s in SIMILARITIES.items()}, "validation frames per variant")

Similarities computed: {'V0': 8, 'V2': 8} validation frames per variant


In [11]:
# 3.4 Regression check of assign_labels
def assign_labels_v15(similarities, margin, background_id=BACKGROUND_ID):
    """assign_labels as in v1.5 (copied unchanged): the reference for margins of zero or above."""
    labels = similarities.argmax(axis=-1)
    best = similarities.max(axis=-1)
    rejected = (labels != background_id) & (best - similarities[..., background_id] < margin)
    return np.where(rejected, background_id, labels)


def background_gap(similarities):
    """d = similarity of the best class other than background - similarity of background (Section 1),
    and the index of that class. Arrays of shape (h, w)."""
    others = similarities.copy()
    others[..., BACKGROUND_ID] = -np.inf
    return others.max(axis=-1) - similarities[..., BACKGROUND_ID], others.argmax(axis=-1)


def variant_confusions(name, margin, gt_key="gt_labels"):
    """Per-frame confusion matrices of one variant at one background margin."""
    return [confusion_matrix(val[gt_key], assign_labels(similarities, margin))
            for val, similarities in zip(VALIDATION_FRAMES, SIMILARITIES[name])]


# (a) Same labels as v1.5 at margin 0 and 0.03, on every validation patch
rows = {}
for name in VARIANTS:
    rows[name] = {f"patches with a different label, margin {margin}":
                  sum(int((assign_labels(s, margin) != assign_labels_v15(s, margin)).sum()) for s in SIMILARITIES[name])
                  for margin in [NO_MARGIN, CURRENT_MARGIN]}
display(pd.DataFrame(rows).T)
print(f"Patches compared per margin and variant: {sum(s[..., 0].size for s in SIMILARITIES['V0'])}")

# (b) Same mIoU as the multi-prototype study (Observation 5.2, values rounded to 3 decimals there)
REPORTED = {  # (variant, ground truth, margin): (10 pieces, arms merged)
    ("V0", "gt_labels", 0.0): (0.626, 0.670), ("V0", "gt_labels", 0.03): (0.673, 0.725),
    ("V0", "gt_tolerant", 0.0): (0.742, 0.811), ("V0", "gt_tolerant", 0.03): (0.753, 0.823),
    ("V2", "gt_labels", 0.0): (0.823, 0.864), ("V2", "gt_labels", 0.03): (0.818, 0.861),
    ("V2", "gt_tolerant", 0.0): (0.823, 0.865), ("V2", "gt_tolerant", 0.03): (0.818, 0.861),
}
rows = []
for (name, gt_key, margin), (reported_10, reported_merged) in REPORTED.items():
    confusions = variant_confusions(name, margin, gt_key)
    obtained_10 = round(frame_miou(confusions, EXCLUDED), 3)
    obtained_merged = round(frame_miou(confusions, EXCLUDED, merge=True), 3)
    rows.append({"variant": name, "metric": gt_key, "margin": margin,
                 "10 pieces": obtained_10, "reported": reported_10,
                 "arms merged": obtained_merged, "reported ": reported_merged,
                 "equal": obtained_10 == reported_10 and obtained_merged == reported_merged})
display(pd.DataFrame(rows))

# (c) Negative margins follow the definition of Section 1; exact ties at d = 0 are counted
rows = {}
for name in VARIANTS:
    gaps = [background_gap(s) for s in SIMILARITIES[name]]
    d_all = np.concatenate([d.ravel() for d, _ in gaps])
    test_margins = np.quantile(d_all[d_all < 0], [0.1, 0.5, 0.9])     # taken from the data: code check only
    mismatches = 0
    for s, (d, best_other) in zip(SIMILARITIES[name], gaps):
        for margin in test_margins:
            expected = np.where(d >= margin, best_other, BACKGROUND_ID)
            mismatches += int((assign_labels(s, margin) != expected).sum())
    rows[name] = {"test margins": np.round(test_margins, 4).tolist(),
                  "patches different from the definition": mismatches,
                  "exact ties d = 0": int((d_all == 0).sum())}
display(pd.DataFrame(rows).T)

,"patches with a different label, margin 0.0","patches with a different label, margin 0.03"
V0,0,0
V2,0,0


Patches compared per margin and variant: 32768


,variant,metric,margin,10 pieces,reported,arms merged,reported,equal
0,V0,gt_labels,0.00,0.626,0.626,0.670,0.670,True
1,V0,gt_labels,0.03,0.673,0.673,0.725,0.725,True
2,V0,gt_tolerant,0.00,0.742,0.742,0.811,0.811,True
3,V0,gt_tolerant,0.03,0.753,0.753,0.823,0.823,True
4,V2,gt_labels,0.00,0.823,0.823,0.864,0.864,True
5,V2,gt_labels,0.03,0.818,0.818,0.861,0.861,True
6,V2,gt_tolerant,0.00,0.823,0.823,0.865,0.865,True
7,V2,gt_tolerant,0.03,0.818,0.818,0.861,0.861,True


,test margins,patches different from the definition,exact ties d = 0
V0,"[-0.2518, -0.2125, -0.1488]",0,0
V2,"[-0.375, -0.2883, -0.1928]",0,0


------------------------------------------------------------------------------------------------------------------------------------------

References Section 3: none. The reference values are those of the multi-prototype study (Observation 4.2 and 5.2).

------------------------------------------------------------------------------------------------------------------------------------------